# 规则引擎模块演示 (Rules Module)

本notebook演示hscredit库中规则引擎模块的功能，包括规则定义、评估和组合。

In [1]:
# 添加项目路径
import sys
import os
sys.path.append('../')

# 初始化设置
from hscredit.utils import init_setting
init_setting(seed=42)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import os

# 加载数据
data_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/hscredit.xlsx'
df = pd.read_excel(data_path)
print(f"数据形状: {df.shape}")
print(f"\n列名: {df.columns.tolist()}")

数据形状: (12448, 85)

列名: ['MOB1', 'MOB2', 'loan_date', 'bj_qy24', 'mobtech80', 'bairong_v1', 'xiaoniu_c3', 'dxm_v6pro', 'bcf_fpv1', 'apply_for_admission_confidence', 'apply_for_admission_score', 'charging_fail_count_12m', 'charging_fail_count_1m', 'charging_fail_count_24m', 'charging_fail_count_3m', 'charging_fail_count_6m', 'consumer_finance_lender_count_12m', 'consumer_finance_lender_count_24m', 'consumer_finance_loan_confidence', 'consumer_finance_loan_credit_line', 'consumer_finance_loan_credit_line_avg', 'consumer_finance_loan_credit_line_max', 'consumer_finance_loan_lender_count', 'consumer_finance_loan_product_count', 'credit_loan_duration', 'hit_consumer_finance_lender_count', 'hit_lender_count', 'hit_network_loan_lender_count', 'last_performance_days', 'lender_count_12m', 'lender_count_1m', 'lender_count_24m', 'lender_count_3m', 'lender_count_6m', 'lender_inquiry_count', 'lender_inquiry_count_1m', 'lender_inquiry_count_3m', 'lender_inquiry_count_6m', 'loan_amt_between_1k_3k_coun

## 1. 导入规则引擎模块

In [2]:
from hscredit.core.rules import Rule, get_columns_from_query, optimize_expr, beautify_expr

print("规则引擎模块导入成功！")

规则引擎模块导入成功！


## 2. 创建简单规则

使用Rule类创建风控规则。

In [3]:
# 选择数值特征进行规则创建
demo_feature = df.select_dtypes(include=[np.number]).columns[0]
print(f"演示特征: {demo_feature}")

# 创建规则: 特征值大于某阈值
mean_val = df[demo_feature].mean()
rule = Rule(f"{demo_feature} > {mean_val}", name=f"高于均值规则")

print(f"规则: {rule.expr}")
print(f"规则名称: {rule.name}")

演示特征: MOB1
规则: MOB1 > 4.079370179948586
规则名称: 高于均值规则


In [4]:
# 规则预测
result = rule.predict(df)
print(f"规则命中样本数: {result.sum()}")
print(f"规则命中率: {result.mean()*100:.2f}%")

规则命中样本数: 885
规则命中率: 7.11%


## 3. 组合规则

使用 & (与)、| (或)、~ (非) 运算符组合规则。

In [5]:
# 选择多个特征
features = df.select_dtypes(include=[np.number]).columns[:3].tolist()
print(f"选择特征: {features}")

# 创建多个规则
rule1 = Rule(f"{features[0]} > {df[features[0]].quantile(0.75)}", name="高值规则")
rule2 = Rule(f"{features[1]} < {df[features[1]].quantile(0.25)}", name="低值规则")

# AND组合
rule_and = rule1 & rule2
print(f"AND规则: {rule_and.expr}")

# OR组合
rule_or = rule1 | rule2
print(f"OR规则: {rule_or.expr}")

# NOT组合
rule_not = ~rule1
print(f"NOT规则: {rule_not.expr}")

选择特征: ['MOB1', 'MOB2', 'bj_qy24']
AND规则: MOB1 > 0.0 & MOB2 < 0.0
OR规则: MOB1 > 0.0 | MOB2 < 0.0
NOT规则: ~(MOB1 > 0.0)


In [6]:
# 评估组合规则
print("规则评估结果:")
for name, r in [('AND', rule_and), ('OR', rule_or), ('NOT', rule_not)]:
    result = r.predict(df)
    print(f"  {name}: 命中{result.sum()}样本, 命中率{result.mean()*100:.2f}%")

规则评估结果:
  AND: 命中0样本, 命中率0.00%
  OR: 命中1610样本, 命中率12.93%
  NOT: 命中10838样本, 命中率87.07%


## 4. 复杂规则表达式

创建包含多个条件的复杂规则。

In [7]:
# 创建复杂规则
rule_complex = Rule(
    f"({features[0]} > {df[features[0]].mean()}) & ({features[1]} < {df[features[1]].median()})",
    name="复杂规则"
)

print(f"复杂规则: {rule_complex.expr}")

result_complex = rule_complex.predict(df)
print(f"命中样本数: {result_complex.sum()}")
print(f"命中率: {result_complex.mean()*100:.2f}%")

复杂规则: (MOB1 > 4.079370179948586) & (MOB2 < 0.0)
命中样本数: 0
命中率: 0.00%


## 5. 规则与目标变量交叉分析

分析规则命中样本的目标分布。

In [8]:
# 规则命中与目标变量交叉分析
rule_result = rule_complex.predict(df)
cross_tab = pd.crosstab(rule_result, df['FPD15'], margins=True)
print("规则命中与FPD15交叉表:")
print(cross_tab)

# 计算规则对坏样本的捕获率
bad_captured = (rule_result & (df['FPD15'] == 1)).sum()
total_bad = (df['FPD15'] == 1).sum()
capture_rate = bad_captured / total_bad * 100
print(f"\n坏样本捕获率: {capture_rate:.2f}%")

规则命中与FPD15交叉表:
FPD15      0    1    All
row_0                   
False  11622  826  12448
All    11622  826  12448

坏样本捕获率: 0.00%


## 6. 规则优化和美化

优化和美化规则表达式。

In [9]:
# 规则表达式优化
expr = "(age > 18) & (income >= 5000) | (credit_score >= 700)"
optimized = optimize_expr(expr)
print(f"原始表达式: {expr}")
print(f"优化后: {optimized}")

# 美化表达式
beautified = beautify_expr(expr)
print(f"美化后: {beautified}")

原始表达式: (age > 18) & (income >= 5000) | (credit_score >= 700)
优化后: (age > 18 & income >= 5000) | credit_score >= 700
美化后: (age > 18 & income >= 5000) | credit_score >= 700


## 7. 规则效果评估报告

使用 `rule.report()` 方法生成详细的规则效果评估报告，包含命中率、坏账率、LIFT值等指标。

In [10]:
# 创建单条规则进行详细评估
rule_eval = Rule(f"{features[0]} > {df[features[0]].quantile(0.75)}", name=f"高值_{features[0]}")

# 使用 rule.report() 生成详细评估报告
# target: 目标变量名称
# desc: 规则描述
# margins: 是否添加合计行
report_df = rule_eval.report(
    datasets=df,
    target='FPD15',
    desc='MOB1高值规则',
    margins=True
)

print("单规则效果评估报告:")
print(report_df.to_string(index=False))

单规则效果评估报告:
规则分类       指标名称     指标含义  分箱  样本总数   样本占比  好样本数  好样本占比  坏样本数  坏样本占比   坏样本率  LIFT值    坏账改善   风险拒绝比    准确率    精确率    召回率   F1分数
验证规则 MOB1 > 0.0 MOB1高值规则  命中  1610 0.1293   784 0.0630   826 0.0664 0.5130 7.7317  0.8707  6.7317 0.9370 0.5130 1.0000 0.6782
验证规则 MOB1 > 0.0 MOB1高值规则 未命中 10838 0.8707 10838 0.8707     0 0.0000 0.0000 0.0000 -0.8707 -1.0000 0.0630 0.0000 0.0000 0.0000
验证规则         合计           合计 12448 1.0000 11622 0.9336   826 0.0664 0.0664 1.0000  0.0000  0.0000 0.9370 0.5130 1.0000 0.6782


## 8. 批量规则评估

对多条规则进行批量评估，使用 rule.report() 获取详细指标。

In [11]:
# 创建多条规则
rules_list = []
for i, col in enumerate(features):
    q75 = df[col].quantile(0.75)
    q25 = df[col].quantile(0.25)
    r_high = Rule(f"{col} > {q75}", name=f"高值_{col}")
    r_low = Rule(f"{col} < {q25}", name=f"低值_{col}")
    rules_list.extend([r_high, r_low])

print(f"创建了 {len(rules_list)} 条规则")

# 批量生成规则评估报告
all_reports = []
for r in rules_list:
    report = r.report(
        datasets=df,
        target='FPD15',
        margins=False
    )
    # 只保留命中行的数据
    hit_report = report[report['分箱'] == '命中'].copy()
    hit_report['规则名称'] = r.name
    all_reports.append(hit_report)

# 合并所有报告
batch_report = pd.concat(all_reports, ignore_index=True)

# 选择关键指标展示
key_metrics = ['规则名称', '样本总数', '样本占比', '坏样本率', 'LIFT值', '坏账改善', '精确率', '召回率']
batch_summary = batch_report[key_metrics].copy()

print("\n批量规则评估结果（命中样本）:")
print(batch_summary.to_string(index=False))

创建了 6 条规则

批量规则评估结果（命中样本）:
      规则名称  样本总数   样本占比   坏样本率  LIFT值    坏账改善    精确率    召回率
   高值_MOB1  1610 0.1293 0.5130 7.7317  0.8707 0.5130 1.0000
   低值_MOB1     0 0.0000 0.0000 0.0000  0.0000 0.0000 0.0000
   高值_MOB2  3107 0.2496 0.2659 4.0064  0.7504 0.2659 1.0000
   低值_MOB2     0 0.0000 0.0000 0.0000  0.0000 0.0000 0.0000
高值_bj_qy24  3082 0.2476 0.0435 0.6552 -0.0854 0.0435 0.1622
低值_bj_qy24  3101 0.2491 0.0896 1.3510  0.0874 0.0896 0.3366


## 9. 保存规则结果

将规则评估结果保存到Excel。

In [13]:
# 保存规则结果 - 使用hscredit的ExcelWriter和DataFrame.save()
from hscredit.report.excel import ExcelWriter

output_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/rule_results.xlsx'

with ExcelWriter(theme_color='2639E9').set_filename(output_path) as writer:
    # 保存单规则详细报告
    report_df.save(writer, sheet_name='单规则报告', title='规则效果评估报告', auto_width=True, condition_cols=['坏样本率', 'LIFT值'])
    # 保存批量评估结果
    batch_summary.save(writer, sheet_name='批量评估', title='批量规则评估结果', auto_width=True, condition_cols=['坏样本率', 'LIFT值'])
    # 保存交叉分析表
    cross_tab.save(writer, sheet_name='交叉分析', title='规则与目标交叉分析', auto_width=True)

print(f"规则结果已保存至: {output_path}")

规则结果已保存至: /Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/rule_results.xlsx
